# OpenMontage — Colab Stage 1 setup
This notebook runs the minimal Stage 1 validation on a Colab GPU runtime.
It installs only the minimal dependencies, configures model caches, downloads required models, and runs the real Stage 1 smoke test.

IMPORTANT: This notebook intentionally runs the real pipeline with OM_LOAD_ADAPTERS=1 and OM_REAL_STRICT=1. If any real adapter or model fails to load, the run will stop and report the failure.


In [ ]:
# 1) Environment and GPU detection
import os, sys
print('Python', sys.version)
try:
  import torch
  print('torch', torch.__version__)
  print('cuda available:', torch.cuda.is_available())
  if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))
    try:
      prop = torch.cuda.get_device_properties(0)
      print('total memory (GB):', round(prop.total_memory / (1024**3),2))
    except Exception as e:
      print('GPU properties not available:', e)
except Exception as e:
  print('torch import failed or not installed:', e)

# show nvidia-smi if available
!nvidia-smi || true

In [ ]:
# 2) Install minimal dependencies (adjusts for CUDA when possible).
# NOTE: Adjust the torch install line if a different CUDA version is needed.
!pip -q install -U pip setuptools wheel
# Install core libs; torch will pick the correct wheel on Colab (CUDA-enabled).
!pip -q install -r requirements-colab.txt
# Ensure ffmpeg binary available
!apt-get -qq update && apt-get -qq install -y ffmpeg || true


In [ ]:
# 3) Prepare project path and PYTHONPATH
# If you cloned the repo into /content/openmontage-colab, adjust accordingly.
repo_root = '/content/openmontage-colab'
os.environ['PYTHONPATH'] = repo_root + ':' + os.environ.get('PYTHONPATH','')
print('PYTHONPATH set to', os.environ['PYTHONPATH'])


In [ ]:
# 4) Hugging Face cache config (optional)
os.environ['HF_HOME'] = os.path.join(repo_root, 'hf_cache')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(repo_root, 'hf_cache')
print('HF caches set to', os.environ['HF_HOME'])


In [ ]:
# 5) Enable strict real-mode and adapter importing for real Stage 1 validation
os.environ['OM_LOAD_ADAPTERS'] = '1'
os.environ['OM_REAL_STRICT'] = '1'
print('OM_LOAD_ADAPTERS, OM_REAL_STRICT set')


In [ ]:
# 6) Download required models (best-effort; keep minimal).
# Adjust the model IDs below if the repo's models.yaml recommends different ids.
from pathlib import Path
models_dir = Path('/content/models')
models_dir.mkdir(exist_ok=True)
print('models_dir:', models_dir)
# Example: prefetch faster-whisper model and qwen3 if available (no-op if not reachable)
try:
  from huggingface_hub import hf_hub_download
  print('hf_hub available')
except Exception as e:
  print('huggingface_hub not available:', e)

print('NOTE: Model downloads can be large. The notebook will stop and report if a model fails to load.')


In [ ]:
# 7) Run Stage 1 smoke script (real run). This will stop if a required adapter or model is missing.
# Make sure the repo is checked out to /content/openmontage-colab or edit repo_root above.
import os, sys, subprocess
os.environ['PYTHONPATH'] = repo_root + ':' + os.environ.get('PYTHONPATH','')
print('Running smoke script with OM_LOAD_ADAPTERS=1 and OM_REAL_STRICT=1')
ret = subprocess.run([sys.executable, 'tools/run_stage1_smoke.py'], cwd=repo_root)
print('smoke script exit code', ret.returncode)


# Notes
- The notebook sets OM_LOAD_ADAPTERS=1 and OM_REAL_STRICT=1 to ensure real adapters are loaded and missing adapters cause an immediate failure.
- If any model is too large for the available VRAM, reduce the model selection in configs/models.yaml or use a smaller variant.
- The smoke script writes a report to projects/smoke-report/artifacts/smoke_report.json inside the repo.
